In [ ]:
# Se carga el dataset ya preprocesado

import pandas as pd

df = pd.read_csv("tsla_processed.csv")
df.head()

,open,high,low,close,volume,retorno_diario,sma_5,sma_20,volatilidad_5,volumen_promedio_5,precio_anterior,precio_futuro
0,0.429893,0.437281,0.436630,0.455516,-0.877272,2.114542,0.276856,0.091213,0.284992,-1.181532,0.270217,650.57
1,0.453347,0.443657,0.482922,0.485687,-1.147637,0.290142,0.330429,0.126750,0.480616,-1.175817,0.455141,780.00
2,0.557587,0.844040,0.613636,0.885793,-0.505550,4.110606,0.469113,0.181108,1.340841,-1.030915,0.485313,887.06
3,1.205094,1.393913,1.124888,1.216747,-0.224783,2.827700,0.669137,0.251366,2.161437,-0.818096,0.885425,734.70
4,1.020375,1.023993,0.711164,0.745757,-0.478977,-3.598348,0.765169,0.294215,1.695284,-0.686598,1.216384,748.96


In [ ]:
# Primer análisis de lo recibido: tamaño del dataset, columnas y estadísticas básicas
print(df.shape)
print(df.columns)
df.describe()

# X son las variables usadas para predecir, y es lo que predeciremos
X = df.drop(columns=["precio_futuro"])
y = df["precio_futuro"]

# Dividir en orden cronológico: entrenamos con el pasado (80%), probamos con lo más recientes (20%)
tam_train = int(len(df) * 0.8)

X_train = X[:tam_train]
X_test = X[tam_train:]
y_train = y[:tam_train]
y_test = y[tam_train:]

print("Filas de entrenamiento:", len(X_train))
print("Filas de prueba:", len(X_test))

(1624, 12)
Index(['open', 'high', 'low', 'close', 'volume', 'retorno_diario', 'sma_5',
       'sma_20', 'volatilidad_5', 'volumen_promedio_5', 'precio_anterior',
       'precio_futuro'],
      dtype='object')
Filas de entrenamiento: 1299
Filas de prueba: 325


In [ ]:
# Random Forest: entrenamos varios árboles y promediamos sus predicciones.
# Se configuran 3 combinaciones para ver qué tan simple o complejo conviene que sea el bosque.

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


configuraciones = [
    {"n_estimators": 50,  "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": None},
]

for config in configuraciones:
    modelo_rf = RandomForestRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=42
    )
    modelo_rf.fit(X_train, y_train)
    pred = modelo_rf.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"n_estimators={config['n_estimators']}, max_depth={config['max_depth']} -> "
          f"MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

n_estimators=50, max_depth=5 -> MAE=10.7699, RMSE=13.9743, R2=0.9429
n_estimators=100, max_depth=10 -> MAE=10.1116, RMSE=13.1188, R2=0.9497
n_estimators=200, max_depth=None -> MAE=10.2738, RMSE=13.3182, R2=0.9482


In [ ]:
from sklearn.svm import SVR

configuraciones_svm = [
    {"kernel": "linear", "C": 1},
    {"kernel": "rbf",    "C": 1},
    {"kernel": "rbf",    "C": 10},
]

for config in configuraciones_svm:
    modelo_svm = SVR(kernel=config["kernel"], C=config["C"])
    modelo_svm.fit(X_train, y_train)
    pred = modelo_svm.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"kernel={config['kernel']}, C={config['C']} -> "
          f"MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

kernel=linear, C=1 -> MAE=9.4435, RMSE=12.0678, R2=0.9574
kernel=rbf, C=1 -> MAE=27.5850, RMSE=35.7853, R2=0.6258
kernel=rbf, C=10 -> MAE=11.5163, RMSE=17.4688, R2=0.9108


In [ ]:
from sklearn.neural_network import MLPRegressor

configuraciones_mlp = [
    {"hidden_layer_sizes": (50,),     "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (100, 50), "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (100, 50), "learning_rate_init": 0.01},
]

for config in configuraciones_mlp:
    modelo_mlp = MLPRegressor(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        learning_rate_init=config["learning_rate_init"],
        max_iter=2000,
        random_state=42
    )
    modelo_mlp.fit(X_train, y_train)
    pred = modelo_mlp.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"capas={config['hidden_layer_sizes']}, lr={config['learning_rate_init']} -> "
          f"MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

capas=(50,), lr=0.001 -> MAE=10.8735, RMSE=13.9901, R2=0.9428
capas=(100, 50), lr=0.001 -> MAE=11.3645, RMSE=14.4447, R2=0.9390
capas=(100, 50), lr=0.01 -> MAE=13.4016, RMSE=16.9383, R2=0.9162


In [ ]:
# Comparamos los 3 modelos con su mejor configuración, para elegir cuál usar en la predicción final

import pandas as pd

resultados = pd.DataFrame({
    "Modelo": ["Random Forest", "SVM (Linear)", "MLP"],
    "Hiperparámetros": ["n_estimators=100, max_depth=10", "kernel=linear, C=1", "capas=(50,), lr=0.001"],
    "MAE": [10.1116, 9.4435, 10.8735],
    "RMSE": [13.1188, 12.0678, 13.9901],
    "R2": [0.9497, 0.9574, 0.9428]
})

resultados

,Modelo,Hiperparámetros,MAE,RMSE,R2
0,Random Forest,"n_estimators=100, max_depth=10",10.1116,13.1188,0.9497
1,SVM (Linear),"kernel=linear, C=1",9.4435,12.0678,0.9574
2,MLP,"capas=(50,), lr=0.001",10.8735,13.9901,0.9428


In [ ]:
# Luego de analizar los datos anteriores, optamos por el modelo mejor entrenado (SVM lineal)
mejor_modelo = SVR(kernel="linear", C=1)
mejor_modelo.fit(X_train, y_train)

def predecir_precio_futuro(ruta_csv_nuevo, modelo, columnas_esperadas):
    """
    Recibe la ruta a un CSV nuevo (con el mismo formato/columnas que el dataset original,
    ya preprocesado y escalado igual que tsla_processed.csv) y devuelve las predicciones.
    """
    datos_nuevos = pd.read_csv(ruta_csv_nuevo)

    # Validación de columnas correctas
    faltantes = set(columnas_esperadas) - set(datos_nuevos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas en el CSV nuevo: {faltantes}")

    X_nuevo = datos_nuevos[columnas_esperadas]
    predicciones = modelo.predict(X_nuevo)
    return predicciones

# Uso del modelo para un nuevo CSV (descomentar cuando se vaya a usar).

# columnas = X_train.columns.tolist()
# resultado = predecir_precio_futuro("datos_nuevos.csv", mejor_modelo, columnas)
# print("Predicciones:")
# print(resultado)

Predicciones:
[635.44363108 637.28906841 732.76842034 ... 397.75576403 391.70744982
 380.67302293]


In [ ]:
# Probando con el test creado anteriormente (alternativa de prueba) llamándolo "Prueba"
X_test.head(5).to_csv("prueba.csv", index=False)

columnas = X_train.columns.tolist()
resultado = predecir_precio_futuro("prueba.csv", mejor_modelo, columnas)
valores_reales = y_test.head(5).values

for i in range(len(resultado)):
    print(f"Predicción: {resultado[i]:.2f}")
    print(f"Valor real: {valores_reales[i]:.2f}")

Predicción: 272.42
Valor real: 282.76
Predicción: 277.12
Valor real: 267.28
Predicción: 260.95
Valor real: 239.43
Predicción: 233.98
Valor real: 233.29
Predicción: 230.25
Valor real: 221.86
